# Verify manifests

Run this interactively against the REAL datasets. 
Purpose:
catch anything the unit tests can't -- unit tests use small synthetic
directory trees; this notebook is the first time the manifest builders
run against the actual GRID/LRS3 data.

**What "looks right" means, concretely (come back to this checklist at the
end):**
- Every manifest has the columns in `manifest_builder.MANIFEST_COLUMNS`,
  in that order.
- `sample_id` is unique within each manifest.
- `video_path`, `audio_path`, `landmark_path` all point at files that
  exist (checked below via `check_file_existence`).
- `duration_sec` is a small positive float (a few seconds), not 0 or NaN.
- `transcript` is non-empty, lowercase, and free of the LRS3 `Text:`
  prefix.
- `check_frame_count_vs_duration` and `check_video_fps` both return empty
  DataFrames (zero problem rows).
- `grid_word_segments.csv`'s frame-boundary conversion is directionally
  correct (checked explicitly near the end of this notebook).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

import importlib, fusion_avsr.data.manifest_builder as bld
importlib.reload(bld)

from fusion_avsr.data.manifest_builder import (
    build_grid_manifest,
    build_grid_word_segments,
    build_lrs3_manifest,
    check_file_existence,
    check_frame_count_vs_duration,
    check_video_fps,
    _align_units_to_frame,
    _parse_grid_align,
    clean_manifest,
    filter_grid_word_segments,
)

## Config

Fill in the real dataset roots below. `LIMIT` caps how many
clips each manifest builder processes -- keep it small (a handful) for
this first interactive pass; only remove it (`LIMIT = None`) once
everything below looks right and you're ready to build the full
manifests.

In [ ]:
# TODO: fill in correct paths
PROJECT_NAME = "your_project_name"  # set once; all dataset paths below derive from it
DATASET_ROOT = Path(f"/scratch/{PROJECT_NAME}/datasets")
LRS3_ROOT = DATASET_ROOT / "lrs3"

grid_all_path = "kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data"
GRID_ROOT = DATASET_ROOT / grid_all_path
GRID_LANDMARKS_ROOT = DATASET_ROOT / "grid_landmarks"  # from scripts/extract_landmarks_grid.py

LRS3_TRAINVAL_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "trainval"
LRS3_TRAINVAL_VIDEO_ROOT = LRS3_ROOT / "ainncy" / "trainval"

LRS3_TEST_VIDEO_ROOT = LRS3_ROOT / "test"
LRS3_TEST_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "test"

AUDIO_OUTPUT_DIR = DATASET_ROOT / "extracted_audio"  # from scripts/extract_audio.sh
MANIFEST_DIR = REPO_ROOT / "manifests"

LIMIT = 5  # a couple of examples from each dataset, not the whole thing

### Audio extraction example

In [4]:
# Run this if you haven't already extracted every audio file. 
# This will help check some functions without the whole thing. 
# But I recommend running the whole audio and landmark extraction beforehand.

#!bash $REPO_ROOT/scripts/extract_audio.sh \
#  $LRS3_ROOT/ainncy/trainval/00j9bKdiOjk \
#  $AUDIO_OUTPUT_DIR \
#  4

#!bash $REPO_ROOT/scripts/extract_audio.sh \
#  $LRS3_ROOT/ainncy/trainval/01GWGmg5jn8 \
#  $AUDIO_OUTPUT_DIR \
#  4

### Running over all videos in LRS3-trainval and GRID

You can find run script for SLURM at `$REPO_ROOT/scripts/extract_audio_slurm.sh`

#### Warning: It could take a lot of time so make sure you have allocated enough time, mem and cpu cores

In [28]:
#!bash $REPO_ROOT/scripts/extract_audio.sh \
#  $LRS3_ROOT/ainncy/trainval \
#  $AUDIO_OUTPUT_DIR \
#  8   # match your requested core count

#!bash $REPO_ROOT/scripts/extract_audio.sh \
#  $GRID_ROOT \
#  $AUDIO_OUTPUT_DIR \
#  8

### Landmark exctraction for GRID example

You can find example full run shell script for SLURM at `$REPO_ROOT/scripts/extract_landmarks_grid.sh`

In [29]:
#!python $REPO_ROOT/scripts/extract_landmarks_grid.py \
#  grid_root=$GRID_ROOT \
#  landmarks_root=$GRID_LANDMARKS_ROOT \
#  device=cpu \
#  limit=5

## Build manifests on a small sample first

Using `limit=LIMIT` here samples just a couple of clips from each
dataset, so this cell runs in seconds instead of the tens of minutes a
full build (~65,000 clips total) would take. This is the fast
iteration loop for checking the manifest builders actually work against
real data before committing to a full run.

In [ ]:
lrs3_trainval_manifest = build_lrs3_manifest(
    video_root=LRS3_TRAINVAL_VIDEO_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TRAINVAL_LANDMARKS_ROOT,
    source="lrs3_trainval",
    output_csv="lrs3_trainval_manifest.csv")#, limit=LIMIT)

print(lrs3_trainval_manifest.shape)
print()
print(lrs3_trainval_manifest.head(2))

In [27]:
grid_manifest = build_grid_manifest(
    GRID_ROOT, GRID_LANDMARKS_ROOT, AUDIO_OUTPUT_DIR, output_csv="grid_manifest.csv")#, limit=LIMIT)
print(grid_manifest.shape)
print()
print(grid_manifest.head(2))

2026-09-20 12:49:00 INFO fusion_avsr.data.manifest_builder: Building GRID manifest from /scratch/your_project_name/datasets/kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data (limit=None)
2026-09-20 12:49:56 INFO fusion_avsr.data.manifest_builder: Built GRID manifest: 33000 clips
2026-09-20 12:50:12 WARNING fusion_avsr.data.manifest_builder: check_file_existence found 105 missing path(s)
2026-09-20 12:50:42 WARNING fusion_avsr.data.manifest_builder: check_frame_count_vs_duration found 4 mismatched clip(s)
2026-09-20 12:50:42 INFO fusion_avsr.data.manifest_builder: Cleaned manifest: 33000 -> 32891 rows (109 dropped)


(33000, 7)

    sample_id                                         video_path  \
0  s10_bbab8n  /scratch/your_project_name/datasets/kaggle_lipne...   
1  s10_bbab9s  /scratch/your_project_name/datasets/kaggle_lipne...   

                                          audio_path  \
0  /scratch/your_project_name/datasets/extracted_au...   
1  /scratch/your_project_name/datasets/extracted_au...   

                                       landmark_path               transcript  \
0  /scratch/your_project_name/datasets/grid_landmar...  bin blue at b eight now   
1  /scratch/your_project_name/datasets/grid_landmar...  bin blue at b nine soon   

   duration_sec source  
0         2.978   grid  
1         2.978   grid  


#### Clean up if there is any missing file

In [29]:
lrs3_trainval_manifest = clean_manifest(lrs3_trainval_manifest, log_path="lrs3_trainval_dropped_clips.log")
print(lrs3_trainval_manifest.shape)

2026-09-20 12:53:20 INFO fusion_avsr.data.manifest_builder: check_file_existence: all referenced paths exist
2026-09-20 12:53:30 INFO fusion_avsr.data.manifest_builder: check_frame_count_vs_duration: all landmark frame counts match
2026-09-20 12:53:30 INFO fusion_avsr.data.manifest_builder: Cleaned manifest: 31982 -> 31982 rows (0 dropped)


(31982, 7)


In [30]:
grid_manifest = clean_manifest(grid_manifest, log_path="grid_dropped_clips.log")
print(grid_manifest.shape)

2026-09-20 12:53:32 WARNING fusion_avsr.data.manifest_builder: check_file_existence found 105 missing path(s)
2026-09-20 12:53:43 WARNING fusion_avsr.data.manifest_builder: check_frame_count_vs_duration found 4 mismatched clip(s)
2026-09-20 12:53:43 INFO fusion_avsr.data.manifest_builder: Cleaned manifest: 33000 -> 32891 rows (109 dropped)


(32891, 7)


### LRS3 test set

In [ ]:
lrs3_test_manifest = build_lrs3_manifest(
    video_root=LRS3_TEST_VIDEO_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TEST_LANDMARKS_ROOT,
    source="lrs3_test",
    output_csv="lrs3_test_manifest.csv")#, limit=LIMIT)
print(lrs3_test_manifest.shape)
print()
print(lrs3_test_manifest.head(2))

In [28]:
lrs3_test_manifest = clean_manifest(lrs3_test_manifest, log_path="lrs3_test_dropped_clips.log")
print(lrs3_test_manifest.shape)

2026-09-20 12:50:42 WARNING fusion_avsr.data.manifest_builder: check_file_existence found 6 missing path(s)
2026-09-20 12:50:42 INFO fusion_avsr.data.manifest_builder: check_frame_count_vs_duration: all landmark frame counts match
2026-09-20 12:50:42 INFO fusion_avsr.data.manifest_builder: Cleaned manifest: 1321 -> 1315 rows (6 dropped)


(1315, 7)


## Spot-check a few rows by hand

TODO: eyeball a couple of rows from each manifest above -- open a
`video_path` or listen to an `audio_path` directly, and confirm the
`transcript` field actually matches what's said in the clip.

## Re-run the consistency checks

These should all come back empty (zero rows). If they don't, that's a
real problem to chase down before trusting the manifest for anything
downstream.

In [42]:
print("check_file_existence (lrs3_trainval):")
display(check_file_existence(lrs3_trainval_manifest))

check_file_existence (lrs3_trainval):


2026-09-19 19:51:14 INFO fusion_avsr.data.manifest_builder: check_file_existence: all referenced paths exist


,sample_id,column,path


In [43]:
print("check_file_existence (grid):")
display(check_file_existence(grid_manifest))

check_file_existence (grid):


2026-09-19 19:51:16 INFO fusion_avsr.data.manifest_builder: check_file_existence: all referenced paths exist


,sample_id,column,path


In [44]:
print("check_file_existence (lrs3_test):")
display(check_file_existence(lrs3_test_manifest))

2026-09-19 19:51:23 INFO fusion_avsr.data.manifest_builder: check_file_existence: all referenced paths exist


check_file_existence (lrs3_test):


,sample_id,column,path


In [45]:
print("check_frame_count_vs_duration (lrs3_trainval):")
display(check_frame_count_vs_duration(lrs3_trainval_manifest))

check_frame_count_vs_duration (lrs3_trainval):


2026-09-19 19:52:26 INFO fusion_avsr.data.manifest_builder: check_frame_count_vs_duration: all landmark frame counts match


,sample_id,landmark_path,expected_frames,actual_frames,diff


In [47]:
print("check_frame_count_vs_duration (grid):")
display(check_frame_count_vs_duration(grid_manifest, tolerance_frames=3))
#problems = check_frame_count_vs_duration(grid_manifest)

#display(problems.head())
#display(problems["diff"].value_counts().sort_index())

check_frame_count_vs_duration (grid):


2026-09-19 19:53:21 INFO fusion_avsr.data.manifest_builder: check_frame_count_vs_duration: all landmark frame counts match


,sample_id,landmark_path,expected_frames,actual_frames,diff


In [48]:
print("check_video_fps (lrs3_trainval):")
display(check_video_fps(lrs3_trainval_manifest))

check_video_fps (lrs3_trainval):


2026-09-19 19:54:50 INFO fusion_avsr.data.manifest_builder: check_video_fps: all video frame rates match


,sample_id,video_path,expected_fps,actual_fps


In [49]:
print("check_video_fps (grid):")
display(check_video_fps(grid_manifest))

check_video_fps (grid):


2026-09-19 19:56:42 INFO fusion_avsr.data.manifest_builder: check_video_fps: all video frame rates match


,sample_id,video_path,expected_fps,actual_fps


## GRID word segments: sanity-check the frame-boundary conversion

`build_grid_word_segments` converts each `.align` timestamp (units of
1/25000 sec) to a frame index via `/25000` then `*25fps`. This is easy to
get subtly wrong (off-by-one, wrong unit, etc.), so before trusting it:
pick one real word segment, and confirm
`(end_frame - start_frame) / 25fps` roughly matches the RAW `.align`
timestamp difference in seconds -- computed independently, without going
through `build_grid_word_segments` itself, as a cross-check.

In [53]:
grid_word_segments = build_grid_word_segments(GRID_ROOT)#, limit=LIMIT)
grid_word_segments = filter_grid_word_segments(grid_word_segments, grid_manifest)
print(grid_word_segments.head(10))

2026-09-19 19:59:26 INFO fusion_avsr.data.manifest_builder: Building GRID word-segment table from /scratch/your_project_name/datasets/kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data (limit=None)
2026-09-19 19:59:51 INFO fusion_avsr.data.manifest_builder: Built GRID word-segment table: 198000 word segments


    sample_id   word  start_frame  end_frame
0  s10_bbab8n    bin           12         16
1  s10_bbab8n   blue           16         21
2  s10_bbab8n     at           21         23
3  s10_bbab8n      b           23         28
4  s10_bbab8n  eight           28         33
5  s10_bbab8n    now           33         39
6  s10_bbab9s    bin           17         22
7  s10_bbab9s   blue           22         26
8  s10_bbab9s     at           26         28
9  s10_bbab9s      b           28         32


In [141]:
# Pick one word segment and cross-check its frame conversion directly
# against the raw .align file, independently of _align_units_to_frame.
import random

sample_row = grid_word_segments.iloc[int(random.uniform(0, len(grid_word_segments)-1))]
sample_id = sample_row["sample_id"]

speaker, clip_id = sample_id.split("_", maxsplit=1)
align_path = GRID_ROOT / f"{speaker}_processed" / "align" / f"{clip_id}.align"

align_rows = _parse_grid_align(align_path)
matching = [r for r in align_rows if r[2] == sample_row["word"]][0]
raw_start_units, raw_end_units, word = matching

raw_duration_sec = (raw_end_units - raw_start_units) / 25000
frame_duration_sec = (sample_row["end_frame"] - sample_row["start_frame"]) / 25

print(f"word={word!r}")
print(f"raw .align duration:  {raw_duration_sec:.3f}s")
print(f"frame-derived duration: {frame_duration_sec:.3f}s")
print(f"difference: {abs(raw_duration_sec - frame_duration_sec):.3f}s (should be well under one frame, ~0.04s)")

word='with'
raw .align duration:  0.120s
frame-derived duration: 0.120s
difference: 0.000s (should be well under one frame, ~0.04s)


## Final checklist

- [ ] All three manifests' columns match `MANIFEST_COLUMNS` exactly.
- [ ] `sample_id` unique within each manifest.
- [ ] `check_file_existence` empty for all three.
- [ ] `check_frame_count_vs_duration` empty for lrs3_trainval and grid.
- [ ] `check_video_fps` empty for lrs3_trainval and grid.
- [ ] GRID word-segment frame conversion cross-check above is within
      ~1 frame (~0.04s) of the raw `.align` duration.
- [ ] Spot-checked transcripts by eye/ear for a couple of rows.

Once every box is checked, remove `limit=LIMIT` (or set `LIMIT = None`)
and re-run to build the full manifests.